# ⚡ Project LabForge: Mobile Practical Lab Record Maker\n### Autonomous Zero-Touch Lab Record Automation for Smartphones & Google Colab\n\n**Why this was built:**\nWhile doing practicals in the lab, we take dozens of screenshots. Later, our teacher gives us the lab record .docx template and we have to manually align every screenshot. Honestly, I am lazy, and whenever I aligned documents manually in Word, they got rejected by the teacher because of broken alignments (hanging step numbers, tiny unreadable dialogs, missing spaces)!\n\nSo I built **LabForge** to solve this once and for all. Now you just upload your teacher's .docx template and screenshots, and this notebook automatically aligns everything with teacher-grade precision and downloads the final .docx and .pdf right to your phone!\n\n💖 *Dedicated to my friends, future students who have to survive lab submissions, and fellow nerds like me!*\n\n---\n**How it works on your phone:**\n1. Run **Step 1 & 2** to set up cloud LibreOffice & Python.\n2. In **Step 3**, tap Choose Files to upload your .docx and screenshots from your phone gallery.\n3. In **Step 4**, enter your free Google Gemini API key to let AI vision map the images.\n4. In **Step 5 & 6**, download your finished, teacher-grade .docx and .pdf straight to your phone!

## Step 1: Install Cloud Dependencies & LibreOffice
Click the **Play (▶)** button on the cell below to set up LibreOffice and Python packages on Google Colab.

In [ ]:
# Install LibreOffice for headless PDF conversion and required Python libraries
!apt-get update -qq
!apt-get install -y -qq libreoffice
!pip install -q python-docx Pillow PyMuPDF lxml google-generativeai
print("✅ Cloud environment ready!")

## Step 2: Clone LabForge Engine
Pull the latest engine scripts and templates from GitHub.

In [ ]:
import os, shutil
if not os.path.exists("labforge"):
    !git clone https://github.com/cid-moosa/labforge.git
    %cd labforge
else:
    %cd labforge
    !git pull

os.makedirs("input_mobile", exist_ok=True)
os.makedirs("output_mobile", exist_ok=True)
print("✅ LabForge engine loaded!")

## Step 3: Upload Your Source `.docx` and Screenshots
Click the **Play (▶)** button below, then tap **Choose Files** to upload:
- Your unformatted source lab `.docx`
- All your screenshots from your phone gallery/files

In [ ]:
from google.colab import files
import glob

# Clean up previous uploads
shutil.rmtree("input_mobile", ignore_errors=True)
os.makedirs("input_mobile", exist_ok=True)

print("📁 Tap 'Choose Files' to upload your .docx document and all screenshots:")
uploaded = files.upload()

docx_file = None
image_files = []

for filename in uploaded.keys():
    target_path = os.path.join("input_mobile", filename)
    shutil.move(filename, target_path)
    if filename.lower().endswith(".docx"):
        docx_file = target_path
    elif filename.lower().endswith((".png", ".jpg", ".jpeg")):
        image_files.append(target_path)

print(f"\n📄 Document: {docx_file}")
print(f"🖼️ Screenshots uploaded: {len(image_files)}")

## Step 4: AI Vision Mapping (VisionTarget)
Enter your free Google Gemini API key to let AI inspect your screenshots and match them to each step in your document.

*(Get your free key in 10 seconds at: [aistudio.google.com/apikey](https://aistudio.google.com/apikey))*

In [ ]:
import getpass
import google.generativeai as genai
import json, re, docx

api_key = getpass.getpass("🔑 Enter your Gemini API key (or paste your key): ")
genai.configure(api_key=api_key)

# Inspect document paragraphs
doc = docx.Document(docx_file)
paragraphs_data = []
for i, p in enumerate(doc.paragraphs):
    t = p.text.strip()
    if t:
        paragraphs_data.append({"paragraph_index": i, "text": t})

print(f"Analyzing {len(paragraphs_data)} paragraphs and {len(image_files)} screenshots with Gemini 1.5 Flash...")

model = genai.GenerativeModel("gemini-1.5-flash")

prompt = f"""
You are VisionTarget, the multimodal vision targeting engine of Project LabForge.
Inspect the uploaded screenshots and pair each definitive action or evidence screenshot to the exact target paragraph_index where it belongs.

CRITICAL RULES:
1. Drop transient loading or splash screens (keep only definitive action or results).
2. If a step has sub-bullets (a, b, c, d), map the screenshot AFTER the final sub-bullet.
3. Map final completed evidence/hashes to the OUTPUT section.

Document paragraphs:
{json.dumps(paragraphs_data, indent=2)}

Screenshots list:
{json.dumps(image_files, indent=2)}

Return STRICT JSON only matching this schema:
{{
  "images": [
    {{
      "after_paragraph_index": <int>,
      "image_path": "<exact_path_string>",
      "caption": "<short descriptive caption>",
      "page_break_before": false
    }}
  ]
}}
"""

response = model.generate_content(prompt)
clean_json = re.search(r'\{.*\}', response.text, re.DOTALL).group(0)
mapping = json.loads(clean_json)

with open("input_mobile/mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, indent=2)

print(f"✅ VisionTarget mapped {len(mapping['images'])} definitive screenshots!")

## Step 5: Execute LabForge Engine (LayoutEngine OOXML Alignment & PDF Export)
Applies the 7 locked teacher-grade OOXML rules, left-aligns steps, scales screenshots inside aspect-ratio bounding boxes, and generates the final PDF.

In [ ]:
out_docx = "output_mobile/Final_Aligned_Record.docx"
out_pdf = "output_mobile/final/Final_Aligned_Record.pdf"

!python scripts/labforge_engine.py --docx "{docx_file}" --mapping "input_mobile/mapping.json" --out "{out_docx}" --out-pdf "{out_pdf}"
!python scripts/verify_pdf.py --pdf "{out_pdf}" --mapping "input_mobile/mapping.json"
print("\n🎉 LAB RECORD COMPLETED & AUDITED!")

## Step 6: Download Your Final Deliverables to Phone
Click the **Play (▶)** button below to download your teacher-grade `.docx` and `.pdf` files directly to your phone.

In [ ]:
print("📥 Downloading final Word document...")
files.download(out_docx)

print("📥 Downloading final PDF document...")
files.download(out_pdf)
print("✅ Download complete! Check your phone's Downloads folder.")